# Mapping Restaurants in Hackney with Google Places API

**A practical introduction to urban data using Python**

**Question:** What kinds of restaurants are found within **750 metres of Hackney Central station**, and how do their **price levels, ratings and cuisines** vary?

We will make **one Places API (New) Nearby Search**, turn the response into a small table and place the restaurants on **one Google map**. That is the entire exercise. Students can later decide what else to investigate.

**What you need:** Python 3, Jupyter, a Google Cloud project with **Places API (New)** and **Maps Static API** enabled, and billing configured. Follow [API_SETUP.md](API_SETUP.md). This exercise makes at most **two billable requests** when run once: one Nearby Search and one map image. Running the cells again can cause additional charges.

**Status:** Draft checked against Google's API documentation; live requests have **not** been run. The 750 m area is a straight-line circle, not a pedestrian walking distance.

## 1 — Define the study area

We use **Hackney Central station** as a clear, repeatable centre rather than saying simply “Hackney”, which covers a much larger area. The coordinates below are approximately **51.5471° N, 0.0561° W**.

A radius of **750 metres** gives us a manageable neighbourhood-scale study. Google Nearby Search (New) returns **up to 20 places per request**. Even if more restaurants exist within the circle, this response is **not an exhaustive inventory**.

The cell below defines our study area and the API query; it does not contact Google.

In [ ]:
CENTRE_LAT = 51.547105     # Hackney Central station (approx.)
CENTRE_LON = -0.056058
RADIUS_METRES = 750
MAX_RESULTS = 20

query = {
    "includedTypes": ["restaurant"],
    "maxResultCount": MAX_RESULTS,
    "rankPreference": "POPULARITY",
    "locationRestriction": {
        "circle": {
            "center": {"latitude": CENTRE_LAT, "longitude": CENTRE_LON},
            "radius": RADIUS_METRES
        }
    }
}

print(f"Search area: {RADIUS_METRES} m around Hackney Central")
print("Maximum returned:", MAX_RESULTS)

## 2 — Get a Google API key

Create a Google Cloud project, enable **Places API (New)** and **Maps Static API**, and configure billing and suitable API-key restrictions as explained in [the setup guide](API_SETUP.md).

Set the key in an environment variable named `GOOGLE_MAPS_API_KEY` **before launching Jupyter**. Do not write real keys into notebook cells or print them. We use the same key for the two services, so its **API restrictions** must allow both. Consider using separate, appropriately restricted keys for production applications.

The next cell checks only whether a key is present.

In [ ]:
import json
import os
from urllib.request import Request, urlopen
from urllib.error import HTTPError, URLError
from urllib.parse import urlencode

api_key = os.getenv("GOOGLE_MAPS_API_KEY")
print("API key available:", bool(api_key))

# Nothing is requested until you explicitly run the next cell.

## 3 — Find restaurants within 750 metres

Google's **Nearby Search (New)** accepts a location, radius and place type. We request only the fields needed for our exercise: **name, coordinates, primary type, price level and rating**. The price level is a broad category, not the price of a specific meal. The primary type can indicate a cuisine (for example `italian_restaurant`) but can also simply say `restaurant`. Missing values are normal.

**Cost:** Requesting `rating` and `priceLevel` puts this query in a more expensive Places data tier than requesting just names and coordinates. Check current pricing and quotas before running. Every run of this cell sends another request.

[Google Nearby Search (New) documentation](https://developers.google.com/maps/documentation/places/web-service/nearby-search)

In [ ]:
FIELD_MASK = (
    "places.id,places.displayName,places.location,"
    "places.primaryType,places.primaryTypeDisplayName,"
    "places.priceLevel,places.rating"
)

def find_restaurants(payload):
    key = os.getenv("GOOGLE_MAPS_API_KEY")
    if not key:
        raise RuntimeError("Set GOOGLE_MAPS_API_KEY before starting Jupyter.")

    request = Request(
        "https://places.googleapis.com/v1/places:searchNearby",
        data=json.dumps(payload).encode("utf-8"),
        headers={
            "Content-Type": "application/json",
            "X-Goog-Api-Key": key,
            "X-Goog-FieldMask": FIELD_MASK
        },
        method="POST"
    )
    try:
        with urlopen(request, timeout=25) as response:
            return json.load(response).get("places", [])
    except HTTPError as error:
        raise RuntimeError(
            f"Places request failed (HTTP {error.code}). "
            "Check API enablement, billing, restrictions and field names."
        ) from None
    except URLError:
        raise RuntimeError("Network request failed.") from None

# One live, potentially billable request:
restaurants_raw = find_restaurants(query)
print("Restaurants returned:", len(restaurants_raw))

## 4 — Turn the response into a dataset

Each result is a Python dictionary. We extract only the attributes needed for our map. **Cuisine** is an approximation derived from Google's place type; it is not a verified menu description. If Google does not provide a specific cuisine, we label it **Unspecified**.

We also give each restaurant a **map label** (1–9, then A–K), so that the map can be read alongside a table. We keep the dataset **in memory** rather than writing Google's Places data into a permanent repository file, in accordance with Google’s restrictions on storing and reusing Places content.

In [ ]:
import string

PRICE_LABELS = {
    "PRICE_LEVEL_FREE": "Free",
    "PRICE_LEVEL_INEXPENSIVE": "Inexpensive",
    "PRICE_LEVEL_MODERATE": "Moderate",
    "PRICE_LEVEL_EXPENSIVE": "Expensive",
    "PRICE_LEVEL_VERY_EXPENSIVE": "Very expensive",
}
MAP_LABELS = string.digits[1:] + string.ascii_uppercase

def cuisine_from_type(primary_type):
    if not primary_type or primary_type == "restaurant":
        return "Unspecified"
    if primary_type.endswith("_restaurant"):
        return primary_type.removesuffix("_restaurant").replace("_", " ").title()
    return primary_type.replace("_", " ").title()

restaurants = []
for index, place in enumerate(restaurants_raw):
    point = place.get("location", {})
    if "latitude" not in point or "longitude" not in point:
        continue
    restaurants.append({
        "map_label": MAP_LABELS[len(restaurants)],
        "name": place.get("displayName", {}).get("text", "Unnamed"),
        "latitude": point["latitude"],
        "longitude": point["longitude"],
        "cuisine": cuisine_from_type(place.get("primaryType")),
        "price": PRICE_LABELS.get(place.get("priceLevel"), "Unknown"),
        "rating": place.get("rating"),
    })

print(f"Ready to map: {len(restaurants)} restaurants")

## 5 — Draw the restaurant map

We use **Google Maps Static API**, so the Places data appears on a Google map, as required by Google's usage policies. **Each numbered marker** corresponds to the restaurant with the same label in the table. Marker **colour** represents Google's relative price category; the table provides **cuisine and rating**.

The 750 m circle is drawn as a simple approximation using geographical coordinates. The map is displayed directly in Jupyter, **not permanently downloaded or committed to GitHub**.

**Costs:** Running the map cell sends one additional Maps Static API request. Google provides logo and map attribution within the image.

[Maps Static API documentation](https://developers.google.com/maps/documentation/maps-static/start)

In [ ]:
import math
from IPython.display import Image, display

PRICE_COLOURS = {
    "Free": "green",
    "Inexpensive": "green",
    "Moderate": "yellow",
    "Expensive": "orange",
    "Very expensive": "red",
    "Unknown": "gray",
}

def radius_boundary(lat, lon, radius_m, steps=48):
    # Approximate a 750 m circle on the local Earth surface.
    earth_radius = 6_371_000
    lat_rad = math.radians(lat)
    points = []
    for i in range(steps + 1):
        angle = 2 * math.pi * i / steps
        delta_lat = (radius_m * math.cos(angle)) / earth_radius
        delta_lon = (radius_m * math.sin(angle)) / (earth_radius * math.cos(lat_rad))
        points.append(
            f"{lat + math.degrees(delta_lat):.6f},"
            f"{lon + math.degrees(delta_lon):.6f}"
        )
    return points

def google_restaurant_map(rows):
    key = os.getenv("GOOGLE_MAPS_API_KEY")
    if not key:
        raise RuntimeError("Set GOOGLE_MAPS_API_KEY first.")

    parameters = [
        ("center", f"{CENTRE_LAT},{CENTRE_LON}"),
        ("zoom", "15"),
        ("size", "640x640"),
        ("scale", "1"),
        ("maptype", "roadmap"),
        ("path", "color:0x335577AA|weight:2|" + "|".join(
            radius_boundary(CENTRE_LAT, CENTRE_LON, RADIUS_METRES)
        )),
        ("markers", f"size:mid|color:blue|{CENTRE_LAT},{CENTRE_LON}"),
    ]
    for row in rows:
        parameters.append((
            "markers",
            f"color:{PRICE_COLOURS[row['price']]}|label:{row['map_label']}|"
            f"{row['latitude']},{row['longitude']}"
        ))
    parameters.append(("key", key))
    url = "https://maps.googleapis.com/maps/api/staticmap?" + urlencode(parameters)
    # The URL contains an API key: never print, save or share it.
    try:
        with urlopen(Request(url), timeout=25) as response:
            if not response.headers.get("Content-Type", "").startswith("image/"):
                raise RuntimeError("Maps Static API returned a non-image response.")
            return response.read()
    except HTTPError as error:
        raise RuntimeError(
            f"Map request failed (HTTP {error.code}). "
            "Check Maps Static API, billing and key restrictions."
        ) from None

# A second live, potentially billable request:
if restaurants:
    display(Image(data=google_restaurant_map(restaurants)))
else:
    print("No restaurant data. Run the Nearby Search cell first.")

## 6 — Read the map

The colour tells us the **relative price category**. The marker label links to the **cuisine and rating** in the accompanying table. This is enough to ask whether different types of restaurants and different price levels appear to cluster within the study area.

Some values may be **Unknown** or **Unspecified**. Do not invent missing information, and do not treat a high rating as a reliable measurement of quality without considering how reviews are collected.

The map shows at most 20 places selected by Google's ranking system. It does **not** show every restaurant within the circle.

In [ ]:
from IPython.display import HTML
from html import escape

def rating_text(value):
    return f"{value:.1f} / 5" if isinstance(value, (int, float)) else "Not available"

table_rows = []
for row in restaurants:
    table_rows.append(
        "<tr>"
        f"<td>{escape(row['map_label'])}</td>"
        f"<td>{escape(row['name'])}</td>"
        f"<td>{escape(row['cuisine'])}</td>"
        f"<td>{escape(row['price'])}</td>"
        f"<td>{escape(rating_text(row['rating']))}</td>"
        "</tr>"
    )

if table_rows:
    display(HTML(
        "<table><thead><tr><th>Map</th><th>Restaurant</th><th>Cuisine</th>"
        "<th>Price level</th><th>Rating</th></tr></thead><tbody>"
        + "".join(table_rows) + "</tbody></table>"
    ))
else:
    print("No restaurant records available.")

### Finished

You have produced **one neighbourhood restaurant dataset in memory, one map and a matching table**. That is the full exercise.

**A question to consider:** Which price categories and cuisines appear in your sample, and how much of that pattern might depend on Google's selection and classification of businesses?

For independent work, students can decide how to develop the map, change the radius or investigate other place types. Those are **not requirements** of this tutorial.

**Important limits and attribution:** Places API (New) returns up to **20 results per Nearby Search**. Google requires Places data shown on a map to appear on a **Google map**; do not substitute an OpenStreetMap or Folium basemap. Google also restricts caching, storing and redistributing Places content. Do not commit the API response, CSV exports or generated image to the public repository without checking the current terms. Keys must never appear in notebook sources or saved outputs. Clear outputs before sharing.

**References:** [Nearby Search (New)](https://developers.google.com/maps/documentation/places/web-service/nearby-search) · [Place fields and billing tiers](https://developers.google.com/maps/documentation/places/web-service/data-fields) · [Maps Static API](https://developers.google.com/maps/documentation/maps-static/start) · [Places API policies](https://developers.google.com/maps/documentation/places/web-service/policies) · [API setup](API_SETUP.md).

**Verification:** The notebook's structure and API request formats are documentation-derived. No billable Google API call has been made during its preparation; desktop testing is required.